## LSTM(Long Short Term Memory) 구조

![LSTM](imgs/LSTM.png)

 LSTM은 은닉층의 메모리 셀에 입력 게이트, 망각 게이트, 출력 게이트를 추가하여 
 
 불필요한 기억을 지우고, 기억해야할 것들을 정한다.

  셀 상태(cell state)라는 값이 추가되었다.($C_t$)

1) 입력 게이트($i$, $g$)

![LSTMInput](imgs/LSTMInput.png)

$i_t = Sigmoid(W_{x_i}x_t + W_{h_i}h_{t - 1} + b_i)$

$g_t = \tanh (W_{x_g}x_t + W_{h_g}h_{t - 1} + b_g)$ 

계산된 $i_t$, $g_t$를 통해

이번에 선택된 기억할 정보의 양을 정하는 게이트다.

$i_t$ : 이번에 들어갈 정보의 양

$g_t$ : 이번에 들어갈 정보의 후보

2) 삭제 게이트

![LSTMDelete](imgs/LSTMDelete.png)

$f_t = Sigmoid(W_{x_f}x_t + W_{h_f}h_{t - 1} + b_f)$

0에 가까울수록 정보를 많이 망각하란 의미이고,

1에 가까울수록 정보를 많이 보존하란 의미이다.

3) Cell State 업데이트

![LSTMCellState](imgs/LSTMCell.png)

$C_t = f_t \cdot C_{t - 1} + i_t \cdot g_t$

4) 출력 게이트

![LSTMOutput](imgs/LSTMOutput.png)

$o_t = Sigmoid(W_{x_o}x_t + W_{h_o}h_{t - 1} + b_o)$

$h_t = o_t \cdot \tanh (C_t)$

LSTM 코드 사용법

In [1]:
import torch, torch.nn as nn
input_dim, hidden_size = 8, 4

nn.LSTM(input_dim, hidden_size ,batch_first=True)  

LSTM(8, 4, batch_first=True)

### GRU(Gated Recurrent Unit)

LSTM에서는 출력, 입력, 삭제 게이트라는 3개의 게이트가 존재하는 반면, 

GRU에서는 업데이트 게이트와 리셋 게이트 두 가지 게이트만이 존재한다. 

GRU는 LSTM보다 학습 속도가 빠르다고 알려져있지만 여러 평가에서 GRU는 LSTM과 비슷한 성능을 보인다고 알려져 있다.

경험적으로 데이터 양이 적을 때는 매개 변수의 양이 적은 GRU가 조금 더 낫고,

데이터 양이 더 많으면 LSTM이 더 낫다고도 한다.

GRU보다 LSTM에 대한 연구나 사용량이 더 많은데,

이는 LSTM이 더 먼저 나온 구조이기 때문이다.

GRU 사용법

In [2]:
nn.GRU(input_dim, hidden_size, batch_first=True)  

GRU(8, 4, batch_first=True)

## LSTM 실습 코드

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# =========================
# 1. 긴 문장 데이터
# =========================
sentence = """
Deep learning models can discover complicated patterns in data, but they do not
truly understand language in the same way humans do. When we train a recurrent
neural network, we give it a sequence of symbols and ask it to predict the next
symbol again and again. Over time, the model adjusts its internal parameters so
that its hidden states become useful summaries of the previous context. Long
short-term memory networks improve ordinary recurrent networks by using gates
that decide what to forget, what to store, and what to reveal as output.
"""

sentence = sentence.lower()

# =========================
# 2. 문자 사전 생성
# =========================
char_set = sorted(list(set(sentence)))
char_dic = {c: i for i, c in enumerate(char_set)}
idx_to_char = {i: c for c, i in char_dic.items()}

dic_size = len(char_dic)

print("문자 개수:", dic_size)
print("문자 사전:", char_dic)

# =========================
# 3. 입력/정답 데이터 생성
# =========================
x_data = [char_dic[c] for c in sentence[:-1]]
y_data = [char_dic[c] for c in sentence[1:]]

x_one_hot = np.eye(dic_size)[x_data]

X = torch.FloatTensor(x_one_hot).unsqueeze(0)  # (1, seq_len, dic_size)
Y = torch.LongTensor(y_data).unsqueeze(0)      # (1, seq_len)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

# =========================
# 4. LSTM 모델 정의
# =========================
class LSTMNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, layers):
        super(LSTMNet, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        out = self.fc(out)
        return out

# =========================
# 5. 모델 생성
# =========================
input_dim = dic_size
hidden_dim = 128
output_dim = dic_size
layers = 2

model = LSTMNet(input_dim, hidden_dim, output_dim, layers)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

# =========================
# 6. 학습
# =========================
epochs = 700

for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(X)  # (1, seq_len, dic_size)

    loss = criterion(
        outputs.reshape(-1, dic_size),
        Y.reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        predicted = outputs.argmax(dim=2).squeeze(0)
        predicted_str = ''.join([idx_to_char[i.item()] for i in predicted])

        print(f"\n[epoch {epoch:03d}] loss: {loss.item():.4f}")
        print(predicted_str[:300])

# =========================
# 7. 최종 예측 확인
# =========================
with torch.no_grad():
    outputs = model(X)
    predicted = outputs.argmax(dim=2).squeeze(0)
    predicted_str = ''.join([idx_to_char[i.item()] for i in predicted])

print("\n================ 최종 결과 ================")
print("\n[입력 문장 앞부분]")
print(sentence[:-1][:500])

print("\n[정답 문장 앞부분]")
print(sentence[1:][:500])

print("\n[모델 예측 앞부분]")
print(predicted_str[:500])

문자 개수: 30
문자 사전: {'\n': 0, ' ': 1, ',': 2, '-': 3, '.': 4, 'a': 5, 'b': 6, 'c': 7, 'd': 8, 'e': 9, 'f': 10, 'g': 11, 'h': 12, 'i': 13, 'j': 14, 'k': 15, 'l': 16, 'm': 17, 'n': 18, 'o': 19, 'p': 20, 'q': 21, 'r': 22, 's': 23, 't': 24, 'u': 25, 'v': 26, 'w': 27, 'x': 28, 'y': 29}
X shape: torch.Size([1, 547, 30])
Y shape: torch.Size([1, 547])

[epoch 000] loss: 3.3915
yyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyy

[epoch 050] loss: 2.8824
ee   te        a      a                  e                                                                                                            a                                                                                                                                                     

[e

언어 모델

[LanguageModel](LanguageModel.ipynb)